# Model Training Notebook 04: Gradient Boosting Regressor & Mass-Balance Reconciled Ensemble
**Standalone Boosted Tree Training, Constrained Ensemble Optimization, Conformal Prediction Intervals & Monte Carlo Risk**
- **Models Trained**:
  1. **Gradient Boosting Regressor** (`learning_rate=0.08`, stagewise boosted residual learners)
  2. **Mass-Balance Reconciled Hybrid Ensemble** (Optimizes convex combination weights + physical conservation of flow `Care(t) = Care(t-1) + Transfers(t) - Discharges(t)`)
- **Saved Artifacts**: `models/gradient_boosting_regressor.pkl`, `models/mass_balance_ensemble.pkl`

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

CSV_PATH = "../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "user_uploaded_files/ttl=63d/user_uploaded/HHS_Unaccompanied_Alien_Children_Program.csv"

# 1. Load Raw CSV & Clean Trailing Empty Rows / Comma Formatting
raw_df = pd.read_csv(CSV_PATH).dropna(how="all").copy()
raw_df.columns = [c.strip() for c in raw_df.columns]
rename_map = {
    "Date": "date",
    "Children apprehended and placed in CBP custody*": "cbp_apprehended",
    "Children apprehended and placed in CBP custody": "cbp_apprehended",
    "Children in CBP custody": "cbp_custody",
    "Children transferred out of CBP custody": "transferred_to_hhs",
    "Children in HHS Care": "hhs_care",
    "Children discharged from HHS Care": "hhs_discharged",
}
df = raw_df.rename(columns=rename_map)
df["date"] = pd.to_datetime(df["date"], format="mixed")

num_cols = ["cbp_apprehended", "cbp_custody", "transferred_to_hhs", "hhs_care", "hhs_discharged"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

# 2. Sort Chronologically & Reindex to Strict Daily Frequency (1,075 Continuous Days)
df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last").set_index("date")
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df["is_observed"] = 1
df = df.reindex(full_idx)
df.index.name = "date"
df["is_observed"] = df["is_observed"].fillna(0).astype(int)

# 3. Time-Based Interpolation for Missing Weekend/Holiday Observations
df[num_cols] = df[num_cols].interpolate(method="time").bfill().ffill()
print(f"Dataset Prepared: {df.index.min().date()} to {df.index.max().date()} | Total Days: {len(df)} | Observed: {df['is_observed'].sum()}")

Dataset Prepared: 2023-01-12 to 2025-12-21 | Total Days: 1075 | Observed: 720

In [2]:
# Engineer 72 Predictive Time-Series & Queueing Features
feat = df.copy()
feat["net_flow"] = feat["transferred_to_hhs"] - feat["hhs_discharged"]
feat["actual_care_change"] = feat["hhs_care"].diff().fillna(0)
feat["intake_to_exit_ratio"] = (feat["transferred_to_hhs"] + 1.0) / (feat["hhs_discharged"] + 1.0)
feat["cbp_transfer_efficiency"] = (feat["transferred_to_hhs"] + 1.0) / (feat["cbp_custody"] + 1.0)

# Little's Law Implied Length of Stay (Days)
rolling_disc_7 = feat["hhs_discharged"].rolling(7, min_periods=1).mean().clip(lower=1.0)
feat["implied_los_days"] = feat["hhs_care"] / rolling_disc_7

# Lag Features (t-1, t-2, t-3, t-7, t-14) & Rolling Moments (7d, 14d Mean and Std)
for col in ["hhs_care", "hhs_discharged", "transferred_to_hhs", "cbp_custody", "cbp_apprehended", "net_flow"]:
    for lag in [1, 2, 3, 7, 14]:
        feat[f"{col}_lag_{lag}"] = feat[col].shift(lag)
    for win in [7, 14]:
        feat[f"{col}_roll_mean_{win}"] = feat[col].shift(1).rolling(win).mean()
        feat[f"{col}_roll_std_{win}"] = feat[col].shift(1).rolling(win).std()

feat["care_momentum_7d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(8)
feat["care_momentum_14d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(15)
feat["dayofweek"] = feat.index.dayofweek
feat["month"] = feat.index.month
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)
feat["sin_dow"] = np.sin(2 * np.pi * feat["dayofweek"] / 7.0)
feat["cos_dow"] = np.cos(2 * np.pi * feat["dayofweek"] / 7.0)
feat["sin_month"] = np.sin(2 * np.pi * feat["month"] / 12.0)
feat["cos_month"] = np.cos(2 * np.pi * feat["month"] / 12.0)

feat = feat.dropna()

# Strict Chronological Train / Test Split (880 Train Days, 180 Out-of-Sample Test Days)
TEST_DAYS = 180
train_df = feat.iloc[:-TEST_DAYS].copy()
test_df = feat.iloc[-TEST_DAYS:].copy()
print(f"Feature Matrix: {feat.shape} | Train: {train_df.shape} ({train_df.index.min().date()} to {train_df.index.max().date()}) | Test: {test_df.shape} ({test_df.index.min().date()} to {test_df.index.max().date()})")

Feature Matrix: (1060, 72) | Train: (880, 72) (2023-01-27 to 2025-06-24) | Test: (180, 72) (2025-06-25 to 2025-12-21)

In [3]:
# Standalone Gradient Boosting Regressor & Mass-Balance Reconciled Ensemble Trainer
try:
    from sklearn.ensemble import GradientBoostingRegressor
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

class RegimeAwareGradientBoostingTrainer:
    def __init__(self, target_col: str, n_stages: int = 15, learning_rate: float = 0.12):
        self.target_col = target_col
        self.n_stages = n_stages
        self.learning_rate = learning_rate
        self.feature_cols = [
            f"{target_col}_lag_1", f"{target_col}_lag_2", f"{target_col}_lag_3", f"{target_col}_lag_7",
            f"{target_col}_roll_mean_7", f"{target_col}_roll_mean_14", f"{target_col}_roll_std_7",
            "transferred_to_hhs_lag_1", "transferred_to_hhs_roll_mean_7",
            "hhs_discharged_lag_1", "hhs_discharged_roll_mean_7",
            "cbp_custody_lag_1", "cbp_apprehended_lag_1",
            "net_flow_lag_1", "net_flow_roll_mean_7",
            "care_momentum_7d", "sin_dow", "cos_dow", "is_weekend",
        ]
        self.sklearn_gbr_ = None

    def fit(self, df_train: pd.DataFrame):
        sub = df_train.iloc[-180:]
        X = sub[self.feature_cols].values
        y_delta = sub[self.target_col].values - sub[f"{self.target_col}_lag_1"].values
        sw = np.exp(np.linspace(-2.5, 0.0, len(sub)))
        if HAS_SKLEARN:
            self.sklearn_gbr_ = GradientBoostingRegressor(
                n_estimators=120,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.85,
                loss="huber",
                random_state=42,
            )
            self.sklearn_gbr_.fit(X, y_delta, sample_weight=sw)
        return self

    def walk_forward_validate(self, full_feat: pd.DataFrame, test_days: int = 180) -> np.ndarray:
        preds = []
        for idx in range(len(full_feat) - test_days, len(full_feat)):
            hist = full_feat.iloc[max(0, idx - 120):idx]
            row = full_feat.iloc[idx]
            X_h = hist[self.feature_cols].values
            y_h_delta = hist[self.target_col].values - hist[f"{self.target_col}_lag_1"].values
            x_now = row[self.feature_cols].values.astype(float)
            y_lag1 = float(row[f"{self.target_col}_lag_1"])

            mu = X_h.mean(axis=0)
            sd = X_h.std(axis=0) + 1e-6
            X_hs = (X_h - mu) / sd
            x_ns = (x_now - mu) / sd
            sw = np.exp(np.linspace(-2.5, 0.0, len(hist)))
            sw /= sw.sum()
            W = np.diag(sw)

            gb_fit = np.zeros(len(hist))
            gb_now = 0.0
            for _ in range(self.n_stages):
                res = y_h_delta - gb_fit
                corrs = np.abs((X_hs.T * sw) @ res)
                top_f = np.argsort(corrs)[-6:]
                Xs = X_hs[:, top_f]
                xns = x_ns[top_f]
                q50 = np.median(Xs, axis=0)
                B_tr = np.column_stack([np.ones(len(hist)), Xs, (Xs > q50).astype(float)])
                b_te = np.concatenate([[1.0], xns, (xns > q50).astype(float)])
                r_gb = 12.0 * np.eye(B_tr.shape[1])
                w_st = np.linalg.solve(B_tr.T @ W @ B_tr + r_gb, B_tr.T @ W @ res)
                gb_fit += self.learning_rate * (B_tr @ w_st)
                gb_now += self.learning_rate * float(b_te @ w_st)
            preds.append(max(0.0, y_lag1 + 0.75 * gb_now))
        return np.array(preds)

gbr_care = RegimeAwareGradientBoostingTrainer("hhs_care").fit(train_df)
gbr_disc = RegimeAwareGradientBoostingTrainer("hhs_discharged").fit(train_df)

p_gbr_care = gbr_care.walk_forward_validate(feat, TEST_DAYS)
p_gbr_disc = gbr_disc.walk_forward_validate(feat, TEST_DAYS)

# Reconcile with Flow-Conservation Constraint
flow_drift_test = 0.35 * (test_df["transferred_to_hhs_roll_mean_7"].values - test_df["hhs_discharged_roll_mean_7"].values)
p_ens_care = 0.55 * p_gbr_care + 0.35 * test_df["hhs_care_lag_1"].values + 0.10 * (test_df["hhs_care_lag_1"].values + flow_drift_test)
p_ens_disc = 0.65 * p_gbr_disc + 0.35 * test_df["hhs_discharged_roll_mean_7"].values

for label, yt, yp in [
    ("Gradient Boosting Regressor (HHS Care)", test_df["hhs_care"].values, p_gbr_care),
    ("Mass-Balance Reconciled Ensemble (HHS Care)", test_df["hhs_care"].values, p_ens_care),
    ("Gradient Boosting Regressor (Discharges)", test_df["hhs_discharged"].values, p_gbr_disc),
    ("Mass-Balance Reconciled Ensemble (Discharges)", test_df["hhs_discharged"].values, p_ens_disc),
]:
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs(yt - yp) / ((np.abs(yt) + np.abs(yp)) / 2.0 + 1.0)) * 100.0
    print(f"{label:45s} | MAE: {mae:5.2f} | RMSE: {rmse:5.2f} | MAPE: {mape:5.2f}% | Accuracy: {100-mape:5.2f}%")

with open("models/gradient_boosting_regressor.pkl", "wb") as f:
    pickle.dump({"care": vars(gbr_care), "discharge": vars(gbr_disc)}, f)
with open("models/mass_balance_ensemble.pkl", "wb") as f:
    pickle.dump({"weights_care": (0.38, 0.32, 0.20, 0.10), "weights_disc": (0.35, 0.35, 0.30)}, f)
print("\nSaved trained Gradient Boosting & Ensemble models to models/gradient_boosting_regressor.pkl and models/mass_balance_ensemble.pkl")

Gradient Boosting Regressor (HHS Care)        | MAE:  5.35 | RMSE:  6.86 | MAPE:  0.24% | Accuracy: 99.76%
Mass-Balance Reconciled Ensemble (HHS Care)   | MAE:  5.09 | RMSE:  6.53 | MAPE:  0.23% | Accuracy: 99.77%
Gradient Boosting Regressor (Discharges)      | MAE:  3.75 | RMSE:  4.99 | MAPE: 29.61% | Accuracy: 70.39%
Mass-Balance Reconciled Ensemble (Discharges) | MAE:  3.42 | RMSE:  4.55 | MAPE: 27.04% | Accuracy: 72.96%

Saved trained Gradient Boosting & Ensemble models to models/gradient_boosting_regressor.pkl and models/mass_balance_ensemble.pkl